# ipynb/loc_compare_nofont_pie.ipynb - 定位对比(饼图,无字体) / Localization compare (pie, no font)

## 项目背景 / Background
定位对比(饼图,无字体)
Localization compare (pie, no font)

## 功能模块 / Modules
- 饼图形式定位对比
- (详见各代码单元 / see code cells)

## 输入 / Inputs
- 上一阶段产物(.npy/.pt/.csv/.json)/ prior-stage outputs
- 内嵌常量与参数 / inline constants and params

## 输出 / Outputs
- 图表(内联显示) / figures (inline)
- 中间变量 / intermediate variables
- 导出文件(.png/.pdf/.csv) / exported files

## 数据流 / Data Flow
1. 加载数据 / Load data
2. 运行分析 / Run analysis
3. 渲染图表 / Render figures
4. 导出 / Export

## 相关文件 / Related Files
- 调用 / Calls: ipynb/loc_compare_nofont.ipynb
- 被调用 / Called by: 报告 / 论文 / report / paper

## 使用示例 / Usage Example
- 在 JupyterLab 中打开 / open in JupyterLab
- 逐单元运行 / run cells sequentially

## 作者 / Author
项目组 / Project Team

## 版本 / Version
1.0



In [3]:
library(showtext)
library(ggplot2)
library(dplyr)
library(tidyr)
library(scales)
library(ggforce)

# 1. 环境配置
showtext_auto()

# 2. 莫兰迪配色方案定义
morandi_colors <- list(
  bg_circle   = "#E9E9E9",  # 底层圆饼：浅灰
  pie_fill    = "#84A59D",  # 饼图填充：莫兰迪绿
  text_dark   = "#5B5B5B",  # 坐标轴文字：深灰
  grid_line   = "#F2F2F2",  # 水平分割线
  stat_mean   = "#B2967D",  # Mean: 褐赭
  stat_median = "#92A8D1",  # Median: 灰蓝
  stat_mode   = "#D6AD60"   # Mode: 芥末黄
)

# 3. 数据处理 (保持原有逻辑，增加参考行数据)
# 2. 数据加载
df_loc <- read.csv("data/rgcnformer_loc.csv", check.names = FALSE)
df_stat <- read.csv("data/statistic_loc.csv")
map_val_to_x <- function(val) {
  idx <- which(k_values >= val)[1]
  if(is.na(idx)) idx <- length(k_values)
  return(idx)
}

k_levels <- c("Top-1", "Top-3", "Top-5", "Top-7", "Top-10", "Top-20", "Top-50")
class_levels <- rev(df_loc$Name)
# 增加一个“参考行”标签
y_labels <- c("Reference", class_levels)

# A. 准备主图数据
plot_data <- df_loc %>%
  pivot_longer(cols = all_of(k_levels), names_to = "Metric", values_to = "Value") %>%
  mutate(
    Metric_num = as.numeric(factor(Metric, levels = k_levels)),
    Class_num = as.numeric(factor(Name, levels = class_levels)) + 1 # 为参考行腾出位置 1
  )

# B. 准备 25%, 50%, 75%, 100% 参考饼图数据 (放在第一行)
ref_vals <- c(0.25, 0.50, 0.75, 1.00)
ref_data <- data.frame(
  Metric_num = 1:4, 
  Value = ref_vals,
  Class_num = 1,
  Name = "Reference"
)

# 合并绘图数据
final_plot_data <- bind_rows(plot_data, ref_data)

# C. 统计标识数据
stat_config <- data.frame(
  StatType = c("Mean", "Median", "Mode"),
  Shape = c(16, 17, 15), 
  Color = c(morandi_colors$stat_mean, morandi_colors$stat_median, morandi_colors$stat_mode)
)

df_markers <- df_stat %>%
  rename(Name = Class) %>%
  select(Name, Mean, Median, Mode) %>%
  pivot_longer(cols = -Name, names_to = "StatType", values_to = "Value") %>%
  mutate(
    Class_num = as.numeric(factor(Name, levels = class_levels)) + 1,
    Base_X = sapply(Value, map_val_to_x)
  ) %>%
  left_join(stat_config, by = "StatType") %>%
  mutate(
    X_pos = Base_X + c(-0.3, 0, 0.3)[match(StatType, c("Mean", "Median", "Mode"))],
    Y_pos = Class_num + 0.4 
  )

# 5. 绘图
p <- ggplot() +
  # A. 背景水平线
  geom_hline(yintercept = seq(0.5, length(y_labels) + 0.5, 1), 
             color = morandi_colors$grid_line, size = 0.5) +
  
  # B. 底层完整圆饼 (r0=0 变为实心)
  geom_arc_bar(data = final_plot_data,
               aes(x0 = Metric_num, y0 = Class_num, r0 = 0, r = 0.35, 
                   start = 0, end = 2*pi),
               fill = morandi_colors$bg_circle, color = NA) +
  
  # C. 进度层饼图 (r0=0)
  geom_arc_bar(data = final_plot_data,
               aes(x0 = Metric_num, y0 = Class_num, r0 = 0, r = 0.35, 
                   start = 0, end = Value * 2 * pi),
               fill = morandi_colors$pie_fill, color = NA) +
  
  # D. 统计标识点 (仅在非参考行显示)
  geom_point(data = df_markers,
             aes(x = X_pos, y = Y_pos, color = StatType, shape = StatType),
             size = 2.5) +
  
  # E. 坐标轴与比例尺
  coord_fixed(ratio = 1) +
  scale_x_continuous(breaks = 1:length(k_levels), labels = k_levels, position = "top") +
  scale_y_continuous(breaks = 1:length(y_labels), labels = y_labels) +
  
  # F. 颜色与形状图例
  scale_color_manual(name = "Statistic", 
                     values = setNames(stat_config$Color, stat_config$StatType)) +
  scale_shape_manual(name = "Statistic", 
                     values = setNames(stat_config$Shape, stat_config$StatType)) +
  
  # G. 主题与图例位置
  theme_minimal() +
  theme(
    axis.title = element_blank(),
    axis.text = element_text(size = 11, color = morandi_colors$text_dark),
    axis.text.y = element_text(face = ifelse(y_labels == "Reference", "italic", "plain")),
    panel.grid = element_blank(),
    # 图例设置：右下角竖排
    legend.position = c(0.92, 0.08), 
    legend.justification = c("right", "bottom"),
    legend.box.background = element_rect(fill = "white", color = "#E0E0E0"),
    legend.direction = "vertical",
    legend.title = element_text(size = 10, face = "bold"),
    plot.margin = margin(20, 20, 20, 20)
  )

# 6. 保存
ggsave("png2/localization_morandi_pie.pdf", p, width = 11, height = 12, device = cairo_pdf)
print(p)

ERROR: [1m[33mError[39m in `mutate()`:[22m
[1m[22m[36mi[39m In argument: `Base_X = sapply(Value, map_val_to_x)`.
[1mCaused by error in `FUN()`:[22m
[33m![39m object 'k_values' not found
